<a href="https://colab.research.google.com/github/juanjimenez-design/central_prestage/blob/main/nb_central_prestage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importar librerías y conexión a drive

In [ ]:
import pandas as pd
import numpy as np
import os
# Conectar con drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)


In [ ]:
def leer_pestaña_rematricula(nombre_pestaña, filas_a_omitir=0):
    """
    Lee una pestaña específica de un Google Sheet y la convierte en DataFrame.

    Argumentos:
    - nombre_pestaña: El nombre exacto de la pestaña (ej. '2026-2A')
    - filas_a_omitir: Cuántas filas hay antes de los encabezados (por defecto 4).
    """
    # 1. Acceder a la pestaña
    worksheet = sh.worksheet(nombre_pestaña)

    # 2. Obtener todos los datos
    data = worksheet.get_all_values()

    # 3. Crear el DataFrame
    # data[filas_a_omitir] es la fila del encabezado (índice 4 para la fila 5)
    # data[filas_a_omitir + 1:] son los datos reales
    df = pd.DataFrame(data[filas_a_omitir + 1:], columns=data[filas_a_omitir])

    print(f"Pestaña '{nombre_pestaña}' cargada con {len(df)} registros.")
    return df

#  Ubicaciones de los archivos

In [ ]:
path_output = 'drive/Shareddrives/Alianzas/3. Data/CENTRAL/Base Pre Stage/output'

In [ ]:
pre_stage_path = 'drive/Shareddrives/Alianzas/3. Data/CENTRAL/Base Pre Stage/input'

path_2026_1c = os.path.join(pre_stage_path, ' Matriculas 2026-1C.gsheet')
path_2026_1d = os.path.join(pre_stage_path, 'Matriculas 2026-1D.gsheet')

path_id_inconcert = 'drive/Shareddrives/Alianzas/3. Data/ID_INCONCERT/CENTRAL/CENTRAL_contact.parquet'
path_id_inconcert_ventas = 'drive/Shareddrives/Alianzas/3. Data/'



# Importación de Datos

# Columnas requeridas

In [ ]:
columns_tokeep = ['Apellidos y Nombres', 'Cedula', 'Correo personal','Correo Institucional', 'Programa', 'Celular', 'Cohorte', 'Matriculado' ]

In [ ]:
# 2026_1C
sh = gc.open(' Matriculas 2026-1C')
df_2026_1c  = leer_pestaña_rematricula('Rep SIS')
df_2026_1c['Cohorte'] = '2026-1C'
df_2026_1c.columns
df_2026_1c.columns = df_2026_1c.columns.str.strip()
df_2026_1c_pre = df_2026_1c.loc[:,columns_tokeep].copy()
df_2026_1c_pre['Correo personal'] = df_2026_1c_pre['Correo personal'].str.strip().str.lower()
df_2026_1c_pre['Correo personal'] = df_2026_1c_pre['Correo personal'].str.replace(' ', '', regex=False).str.lower()



Pestaña 'Rep SIS' cargada con 538 registros.


In [ ]:
# 2026_1D
sh = gc.open('Matriculas 2026-1D')
df_2026_1d  = leer_pestaña_rematricula('Rep SIS')
df_2026_1d['Cohorte'] = '2026-1D'
df_2026_1d.columns = df_2026_1d.columns.str.strip()
df_2026_1d.columns
df_2026_1d_pre = df_2026_1d.loc[:,columns_tokeep].copy()
df_2026_1d_pre['Correo personal'] = df_2026_1d_pre['Correo personal'].str.strip().str.lower()
df_2026_1d_pre['Correo personal'] = df_2026_1d_pre['Correo personal'].str.replace(' ', '', regex=False).str.lower()

Pestaña 'Rep SIS' cargada con 83 registros.


In [ ]:
sh = gc.open('Venta OPM - Academico')
df_id_inconcert_ventas  = leer_pestaña_rematricula('Ventas')
mask_central = df_id_inconcert_ventas['Universidad'] == 'CENTRAL'
df_id_inconcert_ventas_central = df_id_inconcert_ventas[mask_central].copy()
id_inconcert_columns = ['Identificacion','IdLead','Email','Movil']
df_id_inconcert_ventas_central = df_id_inconcert_ventas_central.loc[:,id_inconcert_columns].drop_duplicates().rename(columns= {'IdLead':'id_inconcert'})



Pestaña 'Ventas' cargada con 36405 registros.


In [ ]:
# 1. Quitar espacios en blanco (inicio/fin) y convertir a minúsculas
df_id_inconcert_ventas_central['Email'] = df_id_inconcert_ventas_central['Email'].str.strip().str.lower()

# 2. Opcional: Si quieres quitar espacios intermedios (por si hay errores de dedo)
df_id_inconcert_ventas_central['Email'] = df_id_inconcert_ventas_central['Email'].str.replace(' ', '', regex=False).str.lower()

In [ ]:
# df_id_incncert = pd.read_parquet(path_id_inconcert)
# id_inconcert_columns = ['ID','Email','Phone']
# df_id_incncert_tomerge = df_id_incncert.loc[:,['ID','Email','Phone']].drop_duplicates()

# Proceso

1.   Filtrar sólo 'Matriculados'
2.   Obtener ID INconcert cruzando con id_inconcert







In [ ]:
mask_matriculados = df_2026_1c_pre['Matriculado'] == 'Matriculado'
df_2026_1c_pre = df_2026_1c_pre[mask_matriculados]
mask_matriculados = df_2026_1d_pre['Matriculado'] == 'Matriculado'
df_2026_1d_pre = df_2026_1d_pre[mask_matriculados]

In [ ]:
# 1. Primer intento: Por Identificación
df_1c_cedula = pd.merge(
    df_2026_1c_pre,
    df_id_inconcert_ventas_central.drop(columns=['Email', 'Movil']),
    left_on='Cedula',
    right_on='Identificacion',
    how='left'
)
# 2. Identificar registros que NO cruzaron (Identificacion es NaN)
mask_no_match_1c = df_1c_cedula['Identificacion'].isna()
df_1c_to_retry = df_1c_cedula.loc[mask_no_match_1c,df_2026_1c_pre.columns].copy()
# 3. Segundo intento: Por Email (solo para los que fallaron)
df_1c_email = pd.merge(
    df_1c_to_retry,
    df_id_inconcert_ventas_central.drop(columns=['Identificacion', 'Movil']),
    left_on='Correo personal', # Ajusta el nombre de tu columna de email si es distinto
    right_on='Email',
    how='left'
)

# 4. Concatenar resultados (Cruzados por Cédula + Cruzados por Email)
df_2026_1c_pre_id_inconcert = pd.concat([df_1c_cedula[~mask_no_match_1c], df_1c_email], ignore_index=True)


In [ ]:
# 1. Primer intento: Por Identificación
df_1d_cedula = pd.merge(
    df_2026_1d_pre,
    df_id_inconcert_ventas_central.drop(columns=['Email', 'Movil']),
    left_on='Cedula',
    right_on='Identificacion',
    how='left'
)

# 2. Identificar registros que NO cruzaron
mask_no_match_1d = df_1d_cedula['Identificacion'].isna()
df_1d_to_retry = df_1d_cedula.loc[mask_no_match_1d,df_2026_1d_pre.columns].copy()
# 3. Segundo intento: Por Email
df_1d_email = pd.merge(
    df_1d_to_retry,
    df_id_inconcert_ventas_central.drop(columns=['Identificacion', 'Movil']),
    left_on='Correo personal',
    right_on='Email',
    how='left'
)

# 4. Concatenar
df_2026_1d_pre_id_inconcert = pd.concat([df_1d_cedula[~mask_no_match_1d], df_1d_email], ignore_index=True)

# Validaciones

In [ ]:
columns_tokeep_consolidado = ['id_inconcert'] + columns_tokeep

In [ ]:
df_2026_1c_pre_id_inconcert[df_2026_1c_pre_id_inconcert[columns_tokeep_consolidado].loc[:,'id_inconcert'].isna()]

,Apellidos y Nombres,Cedula,Correo personal,Correo Institucional,Programa,Celular,Cohorte,Matriculado,Identificacion,id_inconcert,Email
281,CARMONA CARMONA OSCAR ALBERTO,18618899,oacarmonac@gmail.com,ocarmonac@ucentral.edu.co,6008 - ADMINISTRACIÓN DE EMPRESAS,3013159930,2026-1C,Matriculado,NaN,NaN,NaN
282,CARVAJAL TANGARIFE CARLOS EDUARDO,9976965,ceduardot@gmail.com,ccarvajalt1@ucentral.edu.co,7010 - MAESTRÍA EN SOSTENIBILIDAD CORPORATIVA,3122897884,2026-1C,Matriculado,NaN,NaN,NaN
283,ORDOÑEZ CRUZ MARIBEL,1081417140,dannamaribel1995@gmail.com,mordonezc1@ucentral.edu.co,7004 - ESPECIALIZACIÓN EN CIENCIAS TRIBUTARIAS,3103332839,2026-1C,Matriculado,NaN,NaN,NaN
284,NAVARRETE HERNANDEZ MIGUEL ANGEL,1016060012,miguelnavarrete03@gmail.com,mnavarreten@ucentral.edu.co,7009 - MAESTRÍA EN ASEGURAMIENTO Y AUDITORÍA D...,3152584977,2026-1C,Matriculado,NaN,NaN,NaN
287,MOLANO SOTO DAVID RENE,1032453226,drms@hotmail.com,dmolanos2@ucentral.edu.co,6007 - DERECHO,3214473878,2026-1C,Matriculado,NaN,NaN,NaN
288,MARROQUIN LINARES JACQUELIN GISSEL,1014188153,jmarroquin0509@gmail.com,jmarroquinl@ucentral.edu.co,7009 - MAESTRÍA EN ASEGURAMIENTO Y AUDITORÍA D...,3177385182,2026-1C,Matriculado,NaN,NaN,NaN
289,MOGOLLON BARRERA PAULA ANDREA,1015480604,paula.andrea.barrera19@gmail.com,pmogollonb@ucentral.edu.co,6007 - DERECHO,3163738735,2026-1C,Matriculado,NaN,NaN,NaN


In [ ]:
df_2026_1d_pre_id_inconcert[df_2026_1d_pre_id_inconcert[columns_tokeep_consolidado].loc[:,'id_inconcert'].isna()]

,Apellidos y Nombres,Cedula,Correo personal,Correo Institucional,Programa,Celular,Cohorte,Matriculado,Identificacion,id_inconcert,Email


# Exportar Datos

In [ ]:
from datetime import datetime
import os

# 1. Definir la ruta base
path_output = "drive/Shareddrives/Alianzas/3. Data/CENTRAL/Base Pre Stage/output"

# 2. Obtener la fecha actual en formato YYYYMMDD
fecha_str = datetime.now().strftime("%Y%m%d")

# 3. Crear las rutas completas con la fecha en el nombre del archivo
path_1c = os.path.join(path_output, f"pre_stage_2026_1C_{fecha_str}.xlsx")
path_1d = os.path.join(path_output, f"pre_stage_2026_1D_{fecha_str}.xlsx")

# 4. Exportar a Excel (puedes cambiar a .to_csv si prefieres)
try:
    df_2026_1c_pre_id_inconcert.loc[:,columns_tokeep_consolidado].to_excel(path_1c, index=False)
    print(f"Exportado exitosamente: {path_1c}")

    df_2026_1d_pre_id_inconcert.loc[:,columns_tokeep_consolidado].to_excel(path_1d, index=False)
    print(f"Exportado exitosamente: {path_1d}")
except Exception as e:
    print(f"Error al exportar: {e}")

Exportado exitosamente: drive/Shareddrives/Alianzas/3. Data/CENTRAL/Base Pre Stage/output/pre_stage_2026_1C_20260508.xlsx
Exportado exitosamente: drive/Shareddrives/Alianzas/3. Data/CENTRAL/Base Pre Stage/output/pre_stage_2026_1D_20260508.xlsx


In [ ]:
df_2026_1c_pre_id_inconcert.shape

(291, 11)

In [ ]:
df_2026_1d_pre_id_inconcert.shape

(49, 11)

In [ ]:
df_2026_1c_pre.shape

(290, 8)

In [ ]:
# Muestra todas las filas donde la cédula se repite
duplicados = df_2026_1d_pre_id_inconcert[df_2026_1d_pre_id_inconcert.duplicated(subset=['Cedula'], keep=False)]

# Ordenar por cédula para ver los grupos de duplicados juntos
duplicados_ordenados = duplicados.sort_values(by='Cedula')

print(duplicados_ordenados)

       Apellidos y Nombres      Cedula Correo personal  \
47  CHIA BOTERO JUAN PABLO  1003527633                   
48  CHIA BOTERO JUAN PABLO  1003527633                   

      Correo Institucional                                         Programa  \
47  jchiab@ucentral.edu.co  7014 - ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS   
48  jchiab@ucentral.edu.co  7014 - ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS   

   Celular  Cohorte  Matriculado Identificacion id_inconcert Email  
47          2026-1D  Matriculado            NaN       156209        
48          2026-1D  Matriculado            NaN         3415        


In [ ]:
df_2026_1d_pre_id_inconcert.columns

Index(['Apellidos y Nombres', 'Cedula', 'Correo personal',
       'Correo Institucional', 'Programa', 'Celular', 'Cohorte', 'Matriculado',
       'Identificacion', 'id_inconcert', 'Email'],
      dtype='object')